In [ ]:
# 1. Import Necessary Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


In [ ]:
# Load the dataset

file_path = "laptop_dataset.csv"
df = pd.read_csv(file_path, low_memory=False)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())


In [ ]:
# Basic dataset information

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))


In [ ]:
# Count missing values

missing_values = df.isnull().sum()
missing_table = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": (missing_values / len(df) * 100).round(2)
}).sort_values("Missing Values", ascending=False)

display(missing_table[missing_table["Missing Values"] > 0])


In [ ]:
# Convert the target price column to numeric where possible

df["Price (Rs)"] = pd.to_numeric(
    df["Price (Rs)"].astype(str).str.replace(r"[^0-9.]", "", regex=True),
    errors="coerce"
)

# Mean imputation for numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mean())

print("Mean imputation completed for numeric columns.")

# Verify remaining missing values
remaining_missing = df.isnull().sum()
display(
    pd.DataFrame({"Remaining Missing Values": remaining_missing})
    .query("`Remaining Missing Values` > 0")
    .sort_values("Remaining Missing Values", ascending=False)
)


In [ ]:
# Outlier counts using the IQR method

numeric_cols = df.select_dtypes(include=np.number).columns

outlier_summary = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append([col, count, lower, upper])

outlier_table = pd.DataFrame(
    outlier_summary,
    columns=["Variable", "Outlier Count", "Lower Bound", "Upper Bound"]
).sort_values("Outlier Count", ascending=False)

display(outlier_table)


In [ ]:
# Visualize target variable before outlier removal

plt.figure(figsize=(10, 5))
sns.boxplot(x=df["Price (Rs)"])
plt.title("Laptop Price Distribution Before Outlier Removal")
plt.xlabel("Price (Rs)")
plt.show()


In [ ]:
# Remove outliers from the target price using IQR

Q1 = df["Price (Rs)"].quantile(0.25)
Q3 = df["Price (Rs)"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

before_rows = len(df)

df = df[
    (df["Price (Rs)"] >= lower_bound) &
    (df["Price (Rs)"] <= upper_bound)
].copy()

after_rows = len(df)

print("Rows before outlier removal:", before_rows)
print("Rows after outlier removal:", after_rows)
print("Outliers removed:", before_rows - after_rows)


In [ ]:
# Visualize price after outlier removal

plt.figure(figsize=(10, 5))
sns.boxplot(x=df["Price (Rs)"])
plt.title("Laptop Price Distribution After Outlier Removal")
plt.xlabel("Price (Rs)")
plt.show()


In [ ]:
# Create a copy for label encoding

encoded_df = df.copy()

categorical_cols = encoded_df.select_dtypes(include=["object", "category"]).columns.tolist()

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    encoded_df[col] = le.fit_transform(encoded_df[col].astype(str))
    label_encoders[col] = le

print("Categorical columns encoded:", len(categorical_cols))
display(encoded_df.head())


In [ ]:
# Correlation with the target

correlation_with_price = encoded_df.corr(numeric_only=True)["Price (Rs)"].sort_values(
    ascending=False
)

display(correlation_with_price.to_frame("Correlation with Price"))


In [ ]:
# Correlation heatmap for numeric/encoded variables

plt.figure(figsize=(16, 12))
corr_matrix = encoded_df.corr(numeric_only=True)

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    linewidths=0.2
)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# Descriptive statistics of the outcome

print("Price statistics:")
display(df["Price (Rs)"].describe().to_frame("Price (Rs)"))


In [ ]:
# Distribution of the outcome variable

plt.figure(figsize=(10, 5))
sns.histplot(df["Price (Rs)"], kde=True, bins=30)
plt.title("Distribution of Laptop Prices")
plt.xlabel("Price (Rs)")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Select useful numeric predictors available in the dataset

candidate_features = [
    "Thickness",
    "Weight",
    "Display Size",
    "Pixel Density",
    "Clock-speed",
    "Cache",
    "Capacity",
    "RAM Speed",
    "SSD Capacity",
    "Power Supply",
    "Refresh Rate",
    "Brightness",
    "Graphics Memory",
    "HDD Capacity",
    "HDD Speed(RPM)",
    "Battery Life",
    "Battery Capacity",
    "Number of Cores"
]

# Keep only columns that actually exist
selected_features = [col for col in candidate_features if col in encoded_df.columns]

# Use encoded values where applicable
X = encoded_df[selected_features].copy()
y = encoded_df["Price (Rs)"].copy()

# Ensure all selected features are numeric
X = X.apply(pd.to_numeric, errors="coerce")

# Mean-impute any conversion-created missing values
X = X.fillna(X.mean())

print("Independent variables:")
print(selected_features)

print("\nX shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# Normalization

minmax_scaler = MinMaxScaler()
X_normalized = pd.DataFrame(
    minmax_scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("Normalized data:")
display(X_normalized.head())


In [ ]:
# Standardization

standard_scaler = StandardScaler()
X_standardized = pd.DataFrame(
    standard_scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("Standardized data:")
display(X_standardized.head())

print("\nApproximate standardized means:")
display(X_standardized.mean().round(4).to_frame("Mean"))

print("Approximate standardized standard deviations:")
display(X_standardized.std().round(4).to_frame("Std"))


In [ ]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X_standardized,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


In [ ]:
# Train Linear Regression model

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Linear Regression Results")
print("-------------------------")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")


In [ ]:
# Actual vs Predicted prices

results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

display(results.head(10))

plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred)
plt.xlabel("Actual Price (Rs)")
plt.ylabel("Predicted Price (Rs)")
plt.title("Actual vs Predicted Laptop Prices")

# Reference line
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.tight_layout()
plt.show()


In [ ]:
# Residual analysis

residuals = y_test - y_pred

plt.figure(figsize=(10, 5))
sns.histplot(residuals, kde=True, bins=30)
plt.title("Distribution of Regression Residuals")
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Frequency")
plt.show()
